# Options and Black-Scholes

Module: Derivatives

## Lesson summary

Options give the holder the right, but not the obligation, to buy or sell an underlying asset. The Black-Scholes model prices European options under idealized assumptions and provides sensitivity measures known as Greeks.

## Learning objectives

By the end of this lesson, students should be able to:

- define call and put option payoffs;
- state the main Black-Scholes assumptions;
- price European calls and puts;
- verify put-call parity;
- calculate and interpret core Greeks.

## Payoffs

For a European call with strike $K$ and terminal stock price $S_T$:

$$
CallPayoff = \max(S_T - K, 0).
$$

For a European put:

$$
PutPayoff = \max(K - S_T, 0).
$$

## Black-Scholes formulas

For a non-dividend-paying asset:

$$
C = S_0N(d_1) - Ke^{-rT}N(d_2),
$$

$$
P = Ke^{-rT}N(-d_2) - S_0N(-d_1),
$$

where:

$$
d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}},
$$

$$
d_2 = d_1 - \sigma\sqrt{T}.
$$

## Python setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

## Black-Scholes implementation

In [ ]:
def black_scholes(S0, K, r, sigma, T, option_type="call"):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    if option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    raise ValueError("option_type must be 'call' or 'put'")


S0 = 100
K = 105
r = 0.06
sigma = 0.25
T = 1

call_price = black_scholes(S0, K, r, sigma, T, "call")
put_price = black_scholes(S0, K, r, sigma, T, "put")
call_price, put_price

## Put-call parity

In [ ]:
left_side = call_price + K * np.exp(-r * T)
right_side = put_price + S0
left_side, right_side, left_side - right_side

## Greeks

In [ ]:
def black_scholes_greeks(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    delta_call = norm.cdf(d1)
    delta_put = norm.cdf(d1) - 1
    gamma = norm.pdf(d1) / (S0 * sigma * np.sqrt(T))
    vega = S0 * norm.pdf(d1) * np.sqrt(T)
    theta_call = (
        -S0 * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
        - r * K * np.exp(-r * T) * norm.cdf(d2)
    )
    theta_put = (
        -S0 * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
        + r * K * np.exp(-r * T) * norm.cdf(-d2)
    )
    rho_call = K * T * np.exp(-r * T) * norm.cdf(d2)
    rho_put = -K * T * np.exp(-r * T) * norm.cdf(-d2)

    return pd.Series(
        {
            "delta_call": delta_call,
            "delta_put": delta_put,
            "gamma": gamma,
            "vega": vega,
            "theta_call": theta_call,
            "theta_put": theta_put,
            "rho_call": rho_call,
            "rho_put": rho_put,
        }
    )


black_scholes_greeks(S0, K, r, sigma, T)

## Payoff diagram

In [ ]:
terminal_prices = np.linspace(50, 160, 200)
payoffs = pd.DataFrame(
    {
        "terminal_price": terminal_prices,
        "call_payoff": np.maximum(terminal_prices - K, 0),
        "put_payoff": np.maximum(K - terminal_prices, 0),
    }
)

ax = payoffs.plot(x="terminal_price", y=["call_payoff", "put_payoff"], figsize=(8, 4))
ax.set_title("European Option Payoffs")
ax.set_xlabel("Terminal stock price")
ax.set_ylabel("Payoff")
ax.grid(True, alpha=0.3)
plt.show()

## Model limitations

- Black-Scholes assumes continuous trading, constant volatility, lognormal dynamics, and frictionless markets.
- Real option markets show smiles, skews, jumps, liquidity effects, and discrete hedging error.
- Greeks are local sensitivities and can change quickly near maturity or around large spot moves.